In [0]:
%sql
--Creating dim_date table 


CREATE OR REPLACE TABLE workspace.media_pipeline.gold_dim_date AS
WITH bounds AS (
  SELECT make_date(year(MIN(report_date)), 1, 1)   AS start_d,
         make_date(year(MAX(report_date)), 12, 31) AS end_d
  FROM workspace.media_pipeline.silver_campaign_performance
),
days AS (
  SELECT explode(sequence(start_d, end_d, INTERVAL 1 DAY)) AS d
  FROM bounds
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d                                       AS full_date,
  year(d)                                 AS year,
  quarter(d)                              AS quarter,
  month(d)                                AS month,
  date_format(d, 'MMMM')                  AS month_name,
  date_format(d, 'EEEE')                  AS day_name,
  dayofweek(d) IN (1, 7)                  AS is_weekend
FROM days

UNION ALL

SELECT 0, CAST(NULL AS DATE), NULL, NULL, NULL, 'Unknown', 'Unknown', NULL;

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
       COUNT_IF(date_key = 0) AS unknown_rows,
       MIN(full_date) AS first_day,
       MAX(full_date) AS last_day
FROM workspace.media_pipeline.gold_dim_date;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.gold_dim_platform AS 
 SELECT DISTINCT
     xxhash64(platform) AS platform_key,
     platform AS platform,
     CASE
        WHEN Platform = 'social_platform' THEN 'Social'
        WHEN Platform = 'daily_campaign_platform' THEN 'Daily Campaign'
        WHEN Platform = 'multichannel_platform' THEN 'Multichannel'
        ELSE platform
    END AS platform_name
     
 FROM workspace.media_pipeline.silver_campaign_performance

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.gold_dim_campaign AS 
 SELECT DISTINCT
     xxhash64(platform,campaign_id) AS campaign_key,
     platform AS platform,
     campaign_id
     
 FROM workspace.media_pipeline.silver_campaign_performance

In [0]:
%sql
SELECT platform, COUNT(*) AS campaigns, COUNT(DISTINCT campaign_key) AS distinct_keys
FROM workspace.media_pipeline.gold_dim_campaign
GROUP BY platform;

In [0]:
%sql
SELECT * FROM workspace.media_pipeline.gold_dim_platform

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.gold_fact_campaign_performance AS
SELECT
  COALESCE(CAST(date_format(report_date, 'yyyyMMdd') AS INT),0) AS date_key,
  xxhash64(platform) AS platform_key,
  xxhash64(platform,campaign_id) AS campaign_key,
  spend, 
  impressions, 
  clicks, 
  conversions,
  is_incomplete, 
  is_view_through, 
  has_funnel_anomaly,
  _source_file
 FROM workspace.media_pipeline.silver_campaign_performance

In [0]:
%sql
SELECT
  COUNT(*)                                   AS total_rows,
  SUM(f.spend)                               AS total_spend,
  COUNT_IF(d.date_key IS NULL)               AS orphan_dates,
  COUNT_IF(p.platform_key IS NULL)           AS orphan_platforms,
  COUNT_IF(c.campaign_key IS NULL)           AS orphan_campaigns
FROM workspace.media_pipeline.gold_fact_campaign_performance f
LEFT JOIN workspace.media_pipeline.gold_dim_date d     ON f.date_key = d.date_key
LEFT JOIN workspace.media_pipeline.gold_dim_platform p ON f.platform_key = p.platform_key
LEFT JOIN workspace.media_pipeline.gold_dim_campaign c ON f.campaign_key = c.campaign_key;

In [0]:
%sql
SELECT
  p.platform_name,
  ROUND(AVG(f.clicks / f.impressions) * 100, 3)        AS avg_of_ctr_pct,
  ROUND(SUM(f.clicks) / SUM(f.impressions) * 100, 3)   AS ctr_from_totals_pct
FROM workspace.media_pipeline.gold_fact_campaign_performance f
JOIN workspace.media_pipeline.gold_dim_platform p
  ON f.platform_key = p.platform_key
WHERE f.impressions > 0
GROUP BY p.platform_name
ORDER BY p.platform_name;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.media_pipeline.gold_vw_platform_kpis AS
SELECT
  p.platform_name,
  -- four totals
  SUM(f.spend)        AS total_spend,
  SUM(f.impressions)  AS total_impressions,
  SUM(f.clicks)       AS total_clicks,
  SUM(f.conversions)  AS total_conversions,
  -- four KPIs
  try_divide(SUM(f.clicks), SUM(f.impressions)) * 100                                     AS ctr_pct,
  try_divide(SUM(f.spend) FILTER (WHERE NOT f.is_incomplete), SUM(f.clicks)) AS cpc,
  try_divide(SUM(f.spend) FILTER (WHERE NOT f.is_incomplete), SUM(f.impressions)) * 1000  AS cpm,
  try_divide(SUM(f.spend) FILTER (WHERE NOT f.is_incomplete), SUM(f.conversions))         AS cpa
FROM workspace.media_pipeline.gold_fact_campaign_performance f
JOIN workspace.media_pipeline.gold_dim_platform p
  ON f.platform_key = p.platform_key
GROUP BY p.platform_name;

In [0]:
%sql
SELECT * FROM workspace.media_pipeline.gold_vw_platform_kpis

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.media_pipeline.gold_vw_billing_reconciliation AS
WITH delivered AS (
  SELECT
    p.platform,
    CASE WHEN f.date_key = 0 THEN 'UNDATED'
         ELSE date_format(d.full_date, 'yyyy-MM') END   AS billing_month,
    SUM(f.spend)                                        AS delivered_spend
  FROM workspace.media_pipeline.gold_fact_campaign_performance f
  JOIN workspace.media_pipeline.gold_dim_platform p ON f.platform_key = p.platform_key
  JOIN workspace.media_pipeline.gold_dim_date d     ON f.date_key = d.date_key
  GROUP BY 1, 2
),
billed AS (
  SELECT
    platform,
    billing_month,
    SUM(amount)                AS billed_amount,
    MAX(is_duplicate_invoice)  AS has_duplicate_invoice
  FROM (SELECT DISTINCT invoice_id, platform, billing_month, amount, is_duplicate_invoice
        FROM workspace.media_pipeline.silver_finance_billing)
  GROUP BY 1, 2
)
SELECT
  COALESCE(d.platform, b.platform)                AS platform,
  COALESCE(d.billing_month, b.billing_month)      AS billing_month,
  d.delivered_spend,
  b.billed_amount,
  b.billed_amount - d.delivered_spend             AS variance,
  COALESCE(b.has_duplicate_invoice, FALSE)        AS has_duplicate_invoice,
  CASE
    WHEN d.delivered_spend IS NULL THEN 'BILLED_NOT_DELIVERED'
    WHEN b.billed_amount IS NULL   THEN 'DELIVERED_NOT_BILLED'
    WHEN ABS(b.billed_amount - d.delivered_spend) <= 1 THEN 'MATCH'
    ELSE 'VARIANCE'
  END                                             AS status
FROM delivered d
FULL OUTER JOIN billed b
  ON d.platform = b.platform AND d.billing_month = b.billing_month;

In [0]:
%sql
SELECT * FROM workspace.media_pipeline.gold_vw_billing_reconciliation
ORDER BY status, platform, billing_month;